# 
e_mannwhitney_raincloud.ipynb\n\n**Purpose:** Non-parametric comparison of Neural Efficiency between groups\nusing Mann-Whitney U test, with raincloud visualization.\n\n**Inputs:**\n- nirs_neural_efficiency.xlsx — NE index per subject × session × ROI\n\n**Outputs:**\n- Mann-Whitney U statistics and raincloud plots (PNG)\n\n**Method:** Mann-Whitney U (two-sided) per ROI; raincloud plot combining\nraw data points, boxplot, and kernel density estimate.\n\n**Library:** scipy, matplotlib, pingouin (Python 3.12)\n\n**Author:** Lucas Gemal (lucasgemal@gmail.com) — IDOR / UFRJ

# Neural Efficiency — Group Comparison
**Mann-Whitney U test + Raincloud Plot**  
Input: fnirs_neural_efficiency.xlsx  
Unit of analysis: mean NE per subject (collapsed across sessions)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

## 1 — Configuration

In [ ]:
# Update this path to match your local data directory
PATH_IN  = r'../data/fnirs_neural_efficiency.xlsx'
# Update this path to match your local data directory
PATH_OUT_FIG   = r'../results/ne_raincloud.png'
# Update this path to match your local data directory
PATH_OUT_EXCEL = r'../results/ne_mannwhitney_results.xlsx'

PALETTE = {
    'acelerado'    : 'darkorange',
    'nao_acelerado': 'steelblue'
}

## 2 — Load and aggregate (mean NE per subject)

In [ ]:
df = pd.read_excel(PATH_IN)

# Mean NE per subject × ROI (collapsed across sessions)
df_subj = (
    df.groupby(['subject', 'group', 'roi'])['neural_efficiency']
    .mean()
    .reset_index()
    .rename(columns={'neural_efficiency': 'ne_mean'})
)

print(f'Rows: {len(df_subj)}  (expected 28 = 14 subjects × 2 ROIs)')
print(df_subj.groupby(['group','roi'])['ne_mean'].describe().round(3))

## 3 — Mann-Whitney U + Effect Size

In [ ]:
def effect_size_label(r):
    r = abs(r)
    if r < 0.3:  return 'small'
    if r < 0.5:  return 'medium'
    return 'large'

results = []

for roi in ['FRONTAL', 'TEMPORAL']:
    sub = df_subj[df_subj['roi'] == roi]
    acc  = sub[sub['group'] == 'acelerado']['ne_mean'].values
    ctrl = sub[sub['group'] == 'nao_acelerado']['ne_mean'].values

    U, p = mannwhitneyu(acc, ctrl, alternative='two-sided')
    n1, n2 = len(acc), len(ctrl)
    r = 1 - (2 * U) / (n1 * n2)  # rank-biserial correlation

    results.append({
        'roi'          : roi,
        'n_acelerado'  : n1,
        'n_nao_acel'   : n2,
        'U'            : U,
        'p_value'      : round(p, 4),
        'effect_size_r': round(r, 3),
        'magnitude'    : effect_size_label(r),
        'significant'  : p < 0.05
    })

df_results = pd.DataFrame(results)

print('\nMann-Whitney Results')
print('='*65)
print(df_results.to_string(index=False))

## 4 — Raincloud Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=True)
rois = ['FRONTAL', 'TEMPORAL']
subtitles = ['Frontal ROI', 'Temporal ROI']
groups = ['acelerado', 'nao_acelerado']
group_labels = {'acelerado': 'Accelerated', 'nao_acelerado': 'Non-Accelerated'}
x_positions = {'acelerado': 0, 'nao_acelerado': 1}

for ax, roi, subtitle in zip(axes, rois, subtitles):
    sub = df_subj[df_subj['roi'] == roi]

    for grp in groups:
        data  = sub[sub['group'] == grp]['ne_mean'].values
        xpos  = x_positions[grp]
        color = PALETTE[grp]

        # ── Violin (half) ────────────────────────────────────
        vp = ax.violinplot(data, positions=[xpos], widths=0.4,
                           showmeans=False, showmedians=False, showextrema=False)
        for body in vp['bodies']:
            # Keep only right half for acelerado, left half for nao_acelerado
            m = np.mean(body.get_paths()[0].vertices[:, 0])
            if grp == 'acelerado':
                body.get_paths()[0].vertices[:, 0] = np.clip(
                    body.get_paths()[0].vertices[:, 0], m, np.inf)
            else:
                body.get_paths()[0].vertices[:, 0] = np.clip(
                    body.get_paths()[0].vertices[:, 0], -np.inf, m)
            body.set_facecolor(color)
            body.set_alpha(0.3)
            body.set_edgecolor(color)

        # ── Boxplot ───────────────────────────────────────────
        bp = ax.boxplot(data, positions=[xpos], widths=0.12,
                        patch_artist=True, showfliers=False,
                        medianprops=dict(color='black', linewidth=2),
                        boxprops=dict(facecolor=color, alpha=0.6),
                        whiskerprops=dict(color=color),
                        capprops=dict(color=color))

        # ── Strip plot (individual points) ───────────────────
        jitter = np.random.uniform(-0.06, 0.06, size=len(data))
        ax.scatter(xpos + jitter, data,
                   color=color, alpha=0.9, s=60, zorder=5,
                   edgecolors='black', linewidths=0.5)

    # ── Stats annotation ─────────────────────────────────────
    row = df_results[df_results['roi'] == roi].iloc[0]
    p_str = f"p={row['p_value']:.3f}" if row['p_value'] >= 0.001 else 'p<0.001'
    sig_str = '✱' if row['significant'] else 'n.s.'
    ax.annotate(
        f"{sig_str}  {p_str}\nr={row['effect_size_r']} ({row['magnitude']})",
        xy=(0.5, 0.97), xycoords='axes fraction',
        ha='center', va='top', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='grey', alpha=0.8)
    )

    ax.axhline(0, linestyle='--', color='grey', linewidth=1, alpha=0.6)
    ax.set_title(subtitle, fontsize=13, fontweight='bold')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Accelerated', 'Non-Accelerated'], fontsize=11)
    ax.set_ylabel('Mean Neural Efficiency', fontsize=11)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

fig.suptitle('Neural Efficiency by Group — Mean Across Sessions',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(PATH_OUT_FIG, dpi=300, bbox_inches='tight')
print(f'-> Figure saved: {PATH_OUT_FIG}')
plt.show()

## 5 — Export results

In [ ]:
with pd.ExcelWriter(PATH_OUT_EXCEL, engine='openpyxl') as writer:
    df_results.to_excel(writer, sheet_name='mannwhitney', index=False)
    df_subj.to_excel(writer,    sheet_name='subject_means', index=False)

print(f'\n✅ Results saved: {PATH_OUT_EXCEL}')
print('   Sheets: mannwhitney | subject_means')